# E03 - Modelo Final

In [1]:
%pip install -q datasets mlflow==3.16.0 scikit-learn pandas numpy

import re
import json
import os
import shutil
import tempfile
import numpy as np
import pandas as pd
import sklearn
import mlflow

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from mlflow import MlflowClient

MLFLOW_URL = "http://ec2-100-26-91-142.compute-1.amazonaws.com:5000"

mlflow.set_tracking_uri(MLFLOW_URL)
mlflow.set_registry_uri(MLFLOW_URL)
mlflow.set_experiment("nlp-lab2-sentiment140")

PROTOCOL_RUN_ID = "5159d4c11e934382ad32d19269f79112"
SELECTED_RUN_ID = "16d0e9a3bcd4468b8b9a1701680ade21"
CONFIGURATION_ID = "CFG_R_TFIDF_UNI_BI"

with open("/opt/ml/metadata/resource-metadata.json") as f:
    meta = json.load(f)

print("ARN:", meta["ResourceArn"])
print("Selected run:", SELECTED_RUN_ID)

Note: you may need to restart the kernel to use updated packages.


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ARN: arn:aws:sagemaker:us-east-1:587058027292:notebook-instance/AnalisisDeSentimientos
Selected run: 16d0e9a3bcd4468b8b9a1701680ade21


In [2]:
dataset = load_dataset(
    "adilbekovich/Sentiment140Twitter",
    revision="b6037e127257d95b9b23d31f78b264b9ebe697fd"
)

train = dataset["train"]
test = dataset["test"]

print("Train:", train.num_rows)
print("Test:", test.num_rows)

assert train.num_rows == 1360000
assert test.num_rows == 240000

Train: 1360000
Test: 240000


In [3]:
def final_preprocess(text):
    text = text.lower()

    # B0
    text = re.sub(r"http\S+|www\S+", "url", text)
    text = re.sub(r"@\w+", "user", text)

    # P_ELONGATION
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # normalización de espacios
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
X_train = train["text"]
y_train = np.array(train["label"])

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    preprocessor=final_preprocess,
    lowercase=False
)

print("Construyendo TF-IDF con 1.360.000 tweets...")

X_train_tfidf = vectorizer.fit_transform(X_train)

print("TF-IDF:", X_train_tfidf.shape)

final_classifier = LogisticRegression()

print("Entrenando Logistic Regression...")

final_classifier.fit(
    X_train_tfidf,
    y_train
)

print("MODELO FINAL ENTRENADO")

Construyendo TF-IDF con 1.360.000 tweets...
TF-IDF: (1360000, 3380663)
Entrenando Logistic Regression...
MODELO FINAL ENTRENADO


In [5]:
print("Train size:", len(y_train))
print("Vocabulary:", len(vectorizer.vocabulary_))
print("Classifier:", final_classifier)

Train size: 1360000
Vocabulary: 3380663
Classifier: LogisticRegression()


In [6]:
X_test = test["text"]
y_test = np.array(test["label"])

print("Evaluando TEST final...")

X_test_tfidf = vectorizer.transform(X_test)

test_predictions = final_classifier.predict(
    X_test_tfidf
)

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro"
)

print("================================")
print("TEST FINAL")
print("Macro-F1:", test_macro_f1)
print("================================")

Evaluando TEST final...
TEST FINAL
Macro-F1: 0.8260640323943584


In [7]:
final_predictions = pd.DataFrame({
    "index": np.arange(len(test_predictions)),
    "text": test["text"],
    "true_label": y_test,
    "predicted_label": test_predictions
})

final_predictions.to_csv(
    "final_test_predictions.csv",
    index=False
)

print("Predicciones guardadas:", len(final_predictions))

Predicciones guardadas: 240000


In [8]:
class Sentiment140FinalModel(mlflow.pyfunc.PythonModel):

    def __init__(self, vectorizer, classifier):
        self.vectorizer = vectorizer
        self.classifier = classifier

    def predict(
        self,
        context,
        model_input: list[str],
        params=None
    ) -> list[str]:

        X = self.vectorizer.transform(model_input)

        predictions = self.classifier.predict(X)

        return [
            "positive" if int(p) == 1 else "negative"
            for p in predictions
        ]


pyfunc_model = Sentiment140FinalModel(
    vectorizer,
    final_classifier
)

print(
    pyfunc_model.predict(
        None,
        [
            "i loved this movie",
            "worst day ever"
        ]
    )
)

['positive', 'negative']


In [10]:
final_config = {
    "preprocessing": {
        "lowercase": True,
        "url": "token:url",
        "mention": "token:user",
        "whitespace": "normalize",
        "stopwords": "keep",
        "negators": [],
        "lemmatize": False,
        "elongation": "normalize",
        "elongation_spec": "reduce_repeated_chars_to_2",
        "emoji": "keep",
        "emoji_spec": None,
        "resources": {},
        "additional": {}
    },

    "representation": {
        "type": "tfidf",
        "ngram_range": [1, 2],
        "library": "sklearn",
        "library_version": sklearn.__version__,
        "spacy_model": None,
        "spacy_model_version": None,
        "parameters": {}
    },

    "classifier": {
        "type": "logistic_regression",
        "library": "sklearn",
        "library_version": sklearn.__version__,
        "parameters": {}
    }
}

In [11]:
# Convertir etiquetas a los nombres exigidos
label_name = {
    0: "negative",
    1: "positive"
}

errors = final_predictions[
    final_predictions["true_label"]
    != final_predictions["predicted_label"]
].copy()

errors["true_label"] = errors["true_label"].map(label_name)
errors["predicted_label"] = errors["predicted_label"].map(label_name)

print("Errores totales:", len(errors))
print("\nErrores por clase real:")
print(errors["true_label"].value_counts())

Errores totales: 41744

Errores por clase real:
true_label
negative    21468
positive    20276
Name: count, dtype: int64


In [12]:
neg_errors = errors[
    errors["true_label"] == "negative"
]

pos_errors = errors[
    errors["true_label"] == "positive"
]

# 10 de cada clase, siempre que existan suficientes
sample_neg = neg_errors.sample(
    n=min(10, len(neg_errors)),
    random_state=42
)

sample_pos = pos_errors.sample(
    n=min(10, len(pos_errors)),
    random_state=42
)

error_sample = pd.concat(
    [sample_neg, sample_pos]
)

# Si por alguna razón quedaron menos de 20, completar aleatoriamente
if len(error_sample) < 20:
    remaining = errors.drop(
        error_sample.index,
        errors="ignore"
    )

    extra = remaining.sample(
        n=20 - len(error_sample),
        random_state=42
    )

    error_sample = pd.concat(
        [error_sample, extra]
    )

# Mezcla determinística
error_sample = error_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Errores seleccionados:", len(error_sample))

error_sample[
    ["index", "text", "true_label", "predicted_label"]
]

Errores seleccionados: 20


,index,text,true_label,predicted_label
0,42486,Well it has been a good weekend! Plenty of res...,negative,positive
1,73332,wants to just sleep until tomorrow night,positive,negative
2,170507,@LittleBitTwistd Isn't Joe the HH?? Holy HOTne...,positive,negative
3,196768,Disgracefull. IKEA have put the price of break...,negative,positive
4,107884,even though much better is bashing my favourit...,negative,positive
5,67897,It's the middle of the night and Twitter is FI...,negative,positive
6,173954,@torilovesbradie hes kgjkcvhjvhkj http://b...,positive,negative
7,186590,@PorchaBaby She's gorgeous!!!! Damn does time ...,negative,positive
8,186153,"@misslion89 Sorry about the retard time thing,...",positive,negative
9,159855,"one year down, like - 1 million more to go",positive,negative


In [13]:
error_sample["category"] = ""

error_sample[
    ["index", "text", "true_label", "predicted_label", "category"]
]

,index,text,true_label,predicted_label,category
0,42486,Well it has been a good weekend! Plenty of res...,negative,positive,
1,73332,wants to just sleep until tomorrow night,positive,negative,
2,170507,@LittleBitTwistd Isn't Joe the HH?? Holy HOTne...,positive,negative,
3,196768,Disgracefull. IKEA have put the price of break...,negative,positive,
4,107884,even though much better is bashing my favourit...,negative,positive,
5,67897,It's the middle of the night and Twitter is FI...,negative,positive,
6,173954,@torilovesbradie hes kgjkcvhjvhkj http://b...,positive,negative,
7,186590,@PorchaBaby She's gorgeous!!!! Damn does time ...,negative,positive,
8,186153,"@misslion89 Sorry about the retard time thing,...",positive,negative,
9,159855,"one year down, like - 1 million more to go",positive,negative,


In [15]:
faltantes = error_sample[
    error_sample["category"].isna()
][["index", "text"]]

faltantes

,index,text
13,23113,@shownalejallah im not hurting anyone why wo...


In [16]:
print(error_sample["index"].tolist())

[42486, 73332, 170507, 196768, 107884, 67897, 173954, 186590, 186153, 159855, 140518, 34703, 112125, 23113, 70452, 36502, 191656, 116386, 53417, 155244]


In [17]:
categories_by_index[23113] = "negation"
categories_by_index.pop(231113, None)

error_sample["category"] = error_sample["index"].map(categories_by_index)

assert len(error_sample) == 20
assert error_sample["category"].notna().all()
assert set(error_sample["category"]).issubset(allowed_categories)

error_sample[
    ["index", "text", "true_label", "predicted_label", "category"]
]

,index,text,true_label,predicted_label,category
0,42486,Well it has been a good weekend! Plenty of res...,negative,positive,mixed
1,73332,wants to just sleep until tomorrow night,positive,negative,other
2,170507,@LittleBitTwistd Isn't Joe the HH?? Holy HOTne...,positive,negative,intensification
3,196768,Disgracefull. IKEA have put the price of break...,negative,positive,intensification
4,107884,even though much better is bashing my favourit...,negative,positive,contrast
5,67897,It's the middle of the night and Twitter is FI...,negative,positive,other
6,173954,@torilovesbradie hes kgjkcvhjvhkj http://b...,positive,negative,informal
7,186590,@PorchaBaby She's gorgeous!!!! Damn does time ...,negative,positive,intensification
8,186153,"@misslion89 Sorry about the retard time thing,...",positive,negative,informal
9,159855,"one year down, like - 1 million more to go",positive,negative,sarcasm


In [18]:
categories_by_index = {
    42486:  "mixed",
    73332:  "other",
    170507: "intensification",
    196768: "intensification",
    107884: "contrast",
    67897:  "other",
    173954: "informal",
    186590: "intensification",
    186153: "informal",
    159855: "sarcasm",
    140518: "other",
    34703:  "elongation",
    112125: "sarcasm",
    231113: "negation",
    70452:  "other",
    36502:  "informal",
    191656: "other",
    116386: "other",
    53417:  "negation",
    155244: "informal",
}

error_sample["category"] = error_sample["index"].map(categories_by_index)

allowed_categories = {
    "negation",
    "intensification",
    "contrast",
    "mixed",
    "emoji",
    "elongation",
    "informal",
    "hashtag",
    "sarcasm",
    "other",
}

assert len(error_sample) == 20
assert error_sample["category"].notna().all()
assert set(error_sample["category"]).issubset(allowed_categories)

error_sample[
    ["index", "text", "true_label", "predicted_label", "category"]
]

AssertionError: 

In [19]:
categories = [
    "mixed",
    "other",
    "intensification",
    "intensification",
    "contrast",
    "other",
    "informal",
    "intensification",
    "informal",
    "sarcasm",
    "other",
    "elongation",
    "sarcasm",
    "negation",
    "other",
    "informal",
    "other",
    "other",
    "negation",
    "informal",
]

assert len(categories) == len(error_sample) == 20

error_sample["category"] = categories

allowed_categories = {
    "negation",
    "intensification",
    "contrast",
    "mixed",
    "emoji",
    "elongation",
    "informal",
    "hashtag",
    "sarcasm",
    "other",
}

assert error_sample["category"].notna().all()
assert set(error_sample["category"]).issubset(allowed_categories)

error_sample[
    ["index", "text", "true_label", "predicted_label", "category"]
]

,index,text,true_label,predicted_label,category
0,42486,Well it has been a good weekend! Plenty of res...,negative,positive,mixed
1,73332,wants to just sleep until tomorrow night,positive,negative,other
2,170507,@LittleBitTwistd Isn't Joe the HH?? Holy HOTne...,positive,negative,intensification
3,196768,Disgracefull. IKEA have put the price of break...,negative,positive,intensification
4,107884,even though much better is bashing my favourit...,negative,positive,contrast
5,67897,It's the middle of the night and Twitter is FI...,negative,positive,other
6,173954,@torilovesbradie hes kgjkcvhjvhkj http://b...,positive,negative,informal
7,186590,@PorchaBaby She's gorgeous!!!! Damn does time ...,negative,positive,intensification
8,186153,"@misslion89 Sorry about the retard time thing,...",positive,negative,informal
9,159855,"one year down, like - 1 million more to go",positive,negative,sarcasm


In [20]:
import os

os.makedirs("reports", exist_ok=True)

error_analysis = error_sample[
    [
        "index",
        "text",
        "true_label",
        "predicted_label",
        "category"
    ]
].copy()

error_analysis.to_csv(
    "reports/error_analysis.csv",
    index=False,
    encoding="utf-8"
)

print("CSV creado correctamente")
print("Filas:", len(error_analysis))
print(error_analysis["category"].value_counts())

CSV creado correctamente
Filas: 20
category
other              6
informal           4
intensification    3
sarcasm            2
negation           2
mixed              1
contrast           1
elongation         1
Name: count, dtype: int64


In [21]:
category_counts = error_analysis["category"].value_counts()

counts_md = "\n".join(
    f"- **{category}**: {count}"
    for category, count in category_counts.items()
)

top_categories = category_counts.head(2).index.tolist()

report_md = f"""# Error Analysis

Se seleccionaron aleatoriamente 20 errores del modelo final utilizando semilla 42, incluyendo errores de ambas clases.

## Frecuencia por categoría

{counts_md}

## Patrones principales

### 1. {top_categories[0]}

Esta categoría fue una de las más frecuentes en la muestra de errores. Los ejemplos muestran que el modelo puede equivocarse cuando la polaridad no está expresada únicamente mediante palabras claramente positivas o negativas, sino que depende del contexto completo de la oración.

### 2. {top_categories[1]}

Este segundo patrón también aparece de forma recurrente. En estos casos, el lenguaje utilizado introduce información que una representación TF-IDF puede tener dificultades para interpretar correctamente, especialmente cuando el sentimiento depende de expresiones no literales, formas informales o contexto.

## Conclusión

El modelo final presenta un buen desempeño global, pero los errores analizados muestran limitaciones frente a fenómenos lingüísticos que requieren mayor comprensión contextual que la disponible mediante una representación TF-IDF clásica.
"""

with open(
    "reports/error_analysis.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(report_md)

print("Markdown creado correctamente")

Markdown creado correctamente


In [22]:
print(os.path.exists("reports/error_analysis.csv"))
print(os.path.exists("reports/error_analysis.md"))

True
True


In [23]:
from mlflow import MlflowClient
import json

client = MlflowClient()

selected_run = client.get_run(SELECTED_RUN_ID)

print("Selected experiment:",
      selected_run.data.tags.get("lab_experiment_id"))
print("Configuration ID:",
      selected_run.data.tags.get("lab_configuration_id"))

selected_config_path = client.download_artifacts(
    SELECTED_RUN_ID,
    "run/configuration.json"
)

with open(selected_config_path, "r", encoding="utf-8") as f:
    selected_config = json.load(f)

assert selected_run.data.tags.get("lab_experiment_id") == "C_LOGREG"
assert selected_run.data.tags.get("lab_configuration_id") == CONFIGURATION_ID
assert selected_config == final_config

print("OK - configuración final coincide con el run seleccionado")

Selected experiment: C_LOGREG
Configuration ID: CFG_R_TFIDF_UNI_BI


OK - configuración final coincide con el run seleccionado


In [24]:
exp = mlflow.get_experiment_by_name("nlp-lab2-sentiment140")

existing_final = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.lab_run_type = 'final'"
)

print("Final runs existentes:", len(existing_final))

for r in existing_final:
    print(r.info.run_id, r.info.status)

Final runs existentes: 0


In [25]:
import tempfile
import shutil
import os
import numpy as np
import sklearn

MODEL_NAME = "sentiment140"

with mlflow.start_run(run_name="final") as run:

    # Tags obligatorios
    mlflow.set_tag("lab_run_type", "final")
    mlflow.set_tag("lab_protocol_run_id", PROTOCOL_RUN_ID)
    mlflow.set_tag(
        "lab_selected_experiment_run_id",
        SELECTED_RUN_ID
    )
    mlflow.set_tag(
        "lab_configuration_id",
        CONFIGURATION_ID
    )
    mlflow.set_tag("lab_member_id", "E03")
    mlflow.set_tag(
        "notebook_arn",
        meta["ResourceArn"]
    )

    # Param obligatorio
    mlflow.log_param(
        "training_size",
        1360000
    )

    # Métrica obligatoria
    mlflow.log_metric(
        "test_macro_f1",
        float(test_macro_f1)
    )

    # Artefactos obligatorios
    with tempfile.TemporaryDirectory() as tmp:

        config_path = os.path.join(
            tmp,
            "configuration.json"
        )

        with open(
            config_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                final_config,
                f,
                indent=2
            )

        mlflow.log_artifact(
            config_path,
            artifact_path="run"
        )

        provenance_path = os.path.join(
            tmp,
            "sagemaker-resource-metadata.json"
        )

        shutil.copy(
            "/opt/ml/metadata/resource-metadata.json",
            provenance_path
        )

        mlflow.log_artifact(
            provenance_path,
            artifact_path="provenance"
        )

    mlflow.log_artifact(
        "reports/error_analysis.csv",
        artifact_path="reports"
    )

    mlflow.log_artifact(
        "reports/error_analysis.md",
        artifact_path="reports"
    )

    # Log + Model Registry
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=pyfunc_model,
        registered_model_name=MODEL_NAME,
        input_example=[
            "i loved this movie",
            "worst day ever"
        ],
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"scikit-learn=={sklearn.__version__}",
            f"numpy=={np.__version__}"
        ]
    )

    final_run_id = run.info.run_id
    model_version = model_info.registered_model_version

print("FINAL RUN ID:", final_run_id)
print("MODEL NAME:", MODEL_NAME)
print("MODEL VERSION:", model_version)
print("TEST MACRO-F1:", test_macro_f1)

2026/09/18 07:55:01 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/09/18 07:55:01 INFO mlflow.models.signature: Running the predict function to generate output based on input example
Successfully registered model 'sentiment140'.
2026/09/18 07:55:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sentiment140, version 1
Created version '1' of model 'sentiment140'.


🏃 View run final at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1/runs/bdb8b088bfbe4c259e03df82d20fc411
🧪 View experiment at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1
FINAL RUN ID: bdb8b088bfbe4c259e03df82d20fc411
MODEL NAME: sentiment140
MODEL VERSION: 1
TEST MACRO-F1: 0.8260640323943584


In [26]:
assert model_version is not None

client = MlflowClient()

client.set_registered_model_alias(
    name="sentiment140",
    alias="champion",
    version=str(model_version)
)

champion = client.get_model_version_by_alias(
    "sentiment140",
    "champion"
)

print("Champion version:", champion.version)
print("Champion run:", champion.run_id)
print("Final run:", final_run_id)

assert champion.run_id == final_run_id

print("OK - sentiment140@champion apunta al run final")

Champion version: 1
Champion run: bdb8b088bfbe4c259e03df82d20fc411
Final run: bdb8b088bfbe4c259e03df82d20fc411
OK - sentiment140@champion apunta al run final


In [27]:
champion_model = mlflow.pyfunc.load_model(
    "models:/sentiment140@champion"
)

test_output = champion_model.predict([
    "i absolutely loved this",
    "this was the worst experience ever"
])

print(test_output)

['positive', 'negative']
